# Rung 2 — Grokking Modular Addition (full P=113 run)

Runs the canonical Nanda et al. (ICLR 2023) grokking setup on a free Colab GPU.
On CPU this takes ~5.5h; on a T4 it takes a few minutes.

**Before running:** Runtime → Change runtime type → GPU.

This notebook clones the repo, then applies a one-line correctness patch to
the train/val split in `exp2_grokking.py` if the cloned copy still has the
bug (splitting by target class `(a+b) % P` instead of by equation `(a, b)`,
which makes some output classes unreachable during training and guarantees
0% validation accuracy no matter how long you train). The patch is a no-op
if the fix is already present — safe to run either way.

In [ ]:
BRANCH = "dev"  # change if you're running from a different branch
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU: Runtime -> Change runtime type -> GPU"

In [ ]:
# Idempotent correctness patch: fixes the train/val split if the cloned
# branch hasn't picked it up yet. Splitting by target class instead of by
# (a, b) equation leaves some output classes with zero training signal --
# not a compute problem, a data-pipeline bug that guarantees 0% val acc.
from pathlib import Path

path = Path("src/experiments/exp2_grokking.py")
src = path.read_text()

buggy = '''    # Split: train on a fraction of modulus values
    train_mod_values = set(
        rng.choice(modulus, size=int(modulus * train_fraction), replace=False)
    )
    train_pairs = [(a, b) for a, b in all_pairs if (a + b) % modulus in train_mod_values]
    val_pairs = [(a, b) for a, b in all_pairs if (a + b) % modulus not in train_mod_values]'''

fixed = '''    # Split: hold out a fraction of (a, b) equations at random so every
    # target class appears in both splits (canonical grokking setup).
    split_idx = int(len(all_pairs) * train_fraction)
    train_pairs = all_pairs[:split_idx]
    val_pairs = all_pairs[split_idx:]'''

if buggy in src:
    path.write_text(src.replace(buggy, fixed))
    print("Patched: train/val split now holds out equations, not target classes.")
else:
    print("No patch needed: the split fix is already present in this branch.")

In [ ]:
# Canonical full run: P=113, 5000 epochs, weight_decay=1.0 (all argparse
# defaults in exp2_grokking.py already match this -- no flags needed beyond
# --save-model to keep the trained checkpoint for downstream rungs).
!python -m src.experiments.exp2_grokking --save-model

In [ ]:
# Zip and download the figures + trained checkpoint.
!zip -r grokking_results.zip figures/exp2_* 

from google.colab import files
files.download("grokking_results.zip")

## After downloading

1. Unzip into `figures/` in your local clone.
2. Check the console output above for: final val accuracy (target: >0.9),
   `k_99_percent` frequencies (target: well under 113, e.g. ~10-20 — that's
   the signature sparse Fourier algorithm the model should have learned).
3. Update `portfolio/RESULTS.md` Rung 2 status and numbers from the real run.
4. If val accuracy is still 0 after this run, the task is no longer a
   compute problem — open an issue with the console output; something in
   the hyperparameters (train_fraction, weight_decay, epochs) needs tuning.